# 📊 Sensitivity analysis — refusal-only vs Ferrando-style labelling

**Goal**: defensive ablation for the paper. The main paper labels an entity as *unknown* when the model gets 0/3 attribute questions correct, regardless of whether it refused or confabulated (we call this *Ferrando-style* — confabulation IS the unknown signal). A reviewer will ask: ``how do AUROCs change under the stricter refusal-only rule (correct==0 AND refused≥1)?''. This notebook answers it directly without re-running the (expensive) labelling step.

**Pipeline**:
1. Load `labelled_raw_v1.json` from HF (1000 entities + their raw answers).
2. Re-classify under both rules.
3. Re-extract residuals at all 64 layers + SAE features at L11/L31/L55 (the only GPU step, ~10 min on A100).
4. Compute single SAE latent, linear probe, diff-of-means AUROC at L11/L31/L55 under both rules with bootstrap CI.
5. Save comparison JSON to `paper_baselines/sensitivity_refusal_vs_ferrando.json`.

In [ ]:
!pip install -q -U transformers accelerate safetensors huggingface_hub datasets scikit-learn matplotlib tqdm

## 1. Config + load model + SAEs

In [ ]:
HF_SAE_REPO   = 'caiovicentino1/qwen36-27b-sae-papergrade'
HF_BASE_MODEL = 'Qwen/Qwen3.6-27B'
SAE_LAYERS    = [11, 31, 55]
ALL_LAYERS    = list(range(64))
D_MODEL       = 5120
D_SAE         = 65_536
K             = 128
PILE_FILTER_THRESHOLD = 0.02
TRAIN_FRAC = 0.7
BOOTSTRAP_N = 1000
SEED = 0

import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
import math, json, random, re
import numpy as np
import torch
random.seed(SEED); torch.manual_seed(SEED); np.random.seed(SEED)
from huggingface_hub import login, hf_hub_download, HfApi
try:
    from google.colab import userdata
    login(token=userdata.get('HF_TOKEN'))
except Exception:
    login()
from transformers import AutoTokenizer, AutoModelForImageTextToText
from safetensors.torch import load_file
import torch.nn.functional as F
from tqdm.auto import tqdm

device = 'cuda'
tok = AutoTokenizer.from_pretrained(HF_BASE_MODEL, trust_remote_code=True)
total_vram_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
max_model_gib  = max(60, int(total_vram_gib * 0.80))
model = AutoModelForImageTextToText.from_pretrained(
    HF_BASE_MODEL, dtype=torch.bfloat16, attn_implementation='sdpa',
    device_map='auto', max_memory={0: f'{max_model_gib}GiB', 'cpu': '40GiB'},
    low_cpu_mem_usage=True, trust_remote_code=True,
)
model.eval()
for p in model.parameters(): p.requires_grad_(False)

class TopKSAE(torch.nn.Module):
    def __init__(self, sd, k):
        super().__init__()
        self.W_enc = torch.nn.Parameter(sd['W_enc'].to(torch.bfloat16), requires_grad=False)
        self.b_enc = torch.nn.Parameter(sd['b_enc'].to(torch.bfloat16), requires_grad=False)
        self.W_dec = torch.nn.Parameter(sd['W_dec'].to(torch.bfloat16), requires_grad=False)
        self.b_dec = torch.nn.Parameter(sd['b_dec'].to(torch.bfloat16), requires_grad=False)
        self.k = k
    def encode(self, x):
        pre = (x - self.b_dec) @ self.W_enc + self.b_enc
        vals, idx = pre.topk(self.k, dim=-1)
        vals = F.relu(vals)
        z = torch.zeros_like(pre); z.scatter_(-1, idx, vals)
        return z

saes = {}
for L in SAE_LAYERS:
    path = hf_hub_download(HF_SAE_REPO, f'sae_L{L}_latest.safetensors')
    saes[L] = TopKSAE(load_file(path), K).to(device).eval()
    print(f'  ✓ SAE L{L}')
print(f'\nready · vram free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB')

## 2. Load labelled checkpoint, re-classify under BOTH rules

In [ ]:
labelled_path = hf_hub_download(repo_id=HF_SAE_REPO, filename='paper_baselines/labelled_raw_v1.json')
labelled_raw = json.load(open(labelled_path))
print(f'loaded {len(labelled_raw)} raw labelled entries')

REFUSAL_PATTERNS = [
    r"\bi (do not|don't) (know|have|recognize|recognise|recall)",
    r"\bi (cannot|can't) (verify|confirm|provide|find|locate)",
    r"\bi'?m not (sure|familiar|aware|certain)",
    r"\bnot (familiar|aware) with",
    r"\b(no|don't have) (specific |any |reliable |verified |confirmed )?(information|data|record|details?|knowledge)",
    r"\b(unable|cannot|can't) (find|locate|provide|identify) (any |specific |reliable )?(information|details?|record)",
    r"\bi (do not|don't) have (any |specific |reliable |verified )?(information|data|details?|knowledge)",
    r"\bi (cannot|can't) (find|recall|remember|locate)",
    r"\b(unknown|unclear|uncertain) to me",
    r"\bthere (is|are|seems) (no|not|little) (widely|publicly|commonly) known",
    r"\bnot (publicly|widely|commonly|well) known",
    r"\bi haven'?t heard (of|about)", r"\bnever heard of",
    r"\bas far as i (know|am aware), (there|i)",
    r"\bunable to provide", r"\bi'?m not familiar",
    r"\bno (wikipedia|reliable|verified|widely-known) (article|entry|source|reference)",
]
REFUSAL_RE = re.compile('|'.join(REFUSAL_PATTERNS), re.IGNORECASE)
def is_refusal(t): return REFUSAL_RE.search(t or '') is not None

def reclassify(rule):
    """rule in ('ferrando', 'refusal_only')"""
    out = []
    for L in labelled_raw:
        refused = sum(1 for a in L['answers'] if is_refusal(a['a']))
        c = L['correct']
        if c >= 2:
            cls = 'known'
        elif rule == 'ferrando':
            cls = 'unknown' if c == 0 else 'middle'
        elif rule == 'refusal_only':
            cls = 'unknown' if (c == 0 and refused >= 1) else 'middle'
        out.append({'entity': L['entity'], 'type': L['type'], 'class': cls,
                    'correct': c, 'refused': refused})
    return out

for rule in ['ferrando', 'refusal_only']:
    rl = reclassify(rule)
    nk = sum(1 for x in rl if x['class'] == 'known')
    nu = sum(1 for x in rl if x['class'] == 'unknown')
    nm = sum(1 for x in rl if x['class'] == 'middle')
    print(f'  {rule:>13}: known={nk}, unknown={nu}, middle={nm} (kept={nk+nu})')

## 3. Forward pass — capture residuals at all 64 layers (~10 min on A100)

We capture residuals for the UNION of entities used by either rule (all entries with correct==0 OR correct>=2 — i.e., known under either rule plus unknown under either rule). This avoids re-running the forward pass twice.

In [ ]:
ferrando = reclassify('ferrando')
refusal  = reclassify('refusal_only')
by_entity_f = {x['entity']: x['class'] for x in ferrando}
by_entity_r = {x['entity']: x['class'] for x in refusal}

# Union: any entity that is known or unknown under EITHER rule
kept_entities = []
for x in ferrando:
    e = x['entity']
    if by_entity_f[e] in ('known', 'unknown') or by_entity_r[e] in ('known', 'unknown'):
        kept_entities.append(x)
n_total = len(kept_entities)
type_label = np.array([x['type'] for x in kept_entities])
y_ferr = np.array([1 if by_entity_f[x['entity']] == 'known' else (0 if by_entity_f[x['entity']] == 'unknown' else -1) for x in kept_entities])
y_ref  = np.array([1 if by_entity_r[x['entity']] == 'known' else (0 if by_entity_r[x['entity']] == 'unknown' else -1) for x in kept_entities])
print(f'union kept: {n_total} entities')
print(f'  ferrando rule: known={int((y_ferr==1).sum())}, unknown={int((y_ferr==0).sum())}, middle={int((y_ferr==-1).sum())}')
print(f'  refusal rule:  known={int((y_ref==1).sum())},  unknown={int((y_ref==0).sum())},  middle={int((y_ref==-1).sum())}')

PROMPT_TEMPLATE = "What can you tell me about '{entity}'?"
def find_entity_last_pos(prompt: str) -> int:
    full_ids = tok(prompt, return_tensors='pt')['input_ids'][0].tolist()
    close_q = tok.encode("'?", add_special_tokens=False)
    n = len(full_ids); m = len(close_q)
    for i in range(n - m, -1, -1):
        if full_ids[i:i+m] == close_q:
            return i - 1
    return n - 2

residuals_all = np.zeros((n_total, 64, D_MODEL), dtype=np.float32)
z_at_sae = {L: np.zeros((n_total, D_SAE), dtype=np.float32) for L in SAE_LAYERS}

with torch.no_grad():
    for i, x in enumerate(tqdm(kept_entities, desc='forward')):
        prompt = PROMPT_TEMPLATE.format(entity=x['entity'])
        ids = tok(prompt, return_tensors='pt')['input_ids'].to(device)
        pos = find_entity_last_pos(prompt)
        out = model(ids, output_hidden_states=True)
        hs = out.hidden_states
        for L in range(64):
            residuals_all[i, L] = hs[L + 1][0, pos].float().cpu().numpy()
        for L in SAE_LAYERS:
            r = torch.tensor(residuals_all[i, L], dtype=torch.bfloat16, device=device)
            z_at_sae[L][i] = saes[L].encode(r.unsqueeze(0))[0].float().cpu().numpy()
        del out
        if i % 25 == 0: torch.cuda.empty_cache()
print('\ndone')

## 4. Pile fire-rate (re-compute, ~5 min)

In [ ]:
from datasets import load_dataset
pile_iter = load_dataset('NeelNanda/pile-10k', split='train', streaming=True)
pile_texts = []
for ex in pile_iter:
    pile_texts.append(ex['text'])
    if len(pile_texts) >= 50: break
fire_count = {L: np.zeros(D_SAE, dtype=np.int64) for L in SAE_LAYERS}
n_pile = 0
with torch.no_grad():
    for txt in tqdm(pile_texts, desc='pile-fire-rate'):
        ids = tok(txt, return_tensors='pt', truncation=True, max_length=512)['input_ids'].to(device)
        out = model(ids, output_hidden_states=True)
        for L in SAE_LAYERS:
            r = out.hidden_states[L + 1][0].to(torch.bfloat16)
            z = saes[L].encode(r).float().cpu().numpy()
            fire_count[L] += (z > 0).sum(axis=0)
        n_pile += ids.shape[1]
        del out
fire_rate = {L: fire_count[L] / n_pile for L in SAE_LAYERS}
for L in SAE_LAYERS:
    print(f'  L{L}: filtered {(fire_rate[L] > PILE_FILTER_THRESHOLD).sum()}/{D_SAE}')

## 5. AUROC under both rules with bootstrap CI

In [ ]:
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

def bootstrap_auroc(y, s, n_boot=BOOTSTRAP_N, seed=0):
    rng = np.random.default_rng(seed); n = len(y); a = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        if len(np.unique(y[idx])) < 2: continue
        try: a.append(roc_auc_score(y[idx], s[idx]))
        except ValueError: continue
    a = np.array(a)
    return float(a.mean()), float(np.percentile(a, 2.5)), float(np.percentile(a, 97.5))

def evaluate(y_label, label_name):
    """Compute SAE/LR/diff-means at L11/L31/L55 with the same train/test split protocol as paper."""
    keep_mask = y_label != -1
    idx_keep = np.where(keep_mask)[0]
    y = y_label[idx_keep]
    np.random.seed(SEED)
    perm = np.arange(len(y)); np.random.shuffle(perm)
    n_tr = int(TRAIN_FRAC * len(y))
    tr_local, te_local = perm[:n_tr], perm[n_tr:]
    tr_global, te_global = idx_keep[tr_local], idx_keep[te_local]
    y_tr, y_te = y_label[tr_global], y_label[te_global]
    print(f'\n=== {label_name} | total kept={len(y)}, train={len(tr_local)} (k={int(y_tr.sum())}, u={int(len(y_tr)-y_tr.sum())}), test={len(te_local)} (k={int(y_te.sum())}, u={int(len(y_te)-y_te.sum())}) ===')
    if len(np.unique(y_te)) < 2 or int(y_te.sum()) < 5 or int(len(y_te) - y_te.sum()) < 5:
        print('  insufficient class diversity in test split, skipping')
        return None
    results = {'rule': label_name, 'n_total': int(len(y)),
               'n_train_known': int(y_tr.sum()), 'n_train_unknown': int(len(y_tr)-y_tr.sum()),
               'n_test_known': int(y_te.sum()), 'n_test_unknown': int(len(y_te)-y_te.sum())}
    # SAE feature
    for L in SAE_LAYERS:
        Z = z_at_sae[L]
        Z_tr = Z[tr_global]
        Zk_tr = Z_tr[y_tr == 1]; Zu_tr = Z_tr[y_tr == 0]
        mu_k, mu_u = Zk_tr.mean(0), Zu_tr.mean(0)
        sd = np.sqrt((Zk_tr.std(0)**2 + Zu_tr.std(0)**2) / 2) + 1e-9
        sep = (mu_k - mu_u) / sd
        pile_mask = fire_rate[L] <= PILE_FILTER_THRESHOLD
        sep_filtered = np.where(pile_mask, sep, 0.0)
        top_feats = np.argsort(-np.abs(sep_filtered))[:100]
        best = {'feat': None, 'auroc_pt': -1}
        Z_te = Z[te_global]
        for feat in top_feats:
            score = Z_te[:, feat]
            try: auc = roc_auc_score(y_te, score)
            except: continue
            if auc < 0.5: auc = 1 - auc; score = -score
            if auc > best['auroc_pt']:
                best = {'feat': int(feat), 'auroc_pt': float(auc), 'score': score, 'sep': float(sep_filtered[feat])}
        m, lo, hi = bootstrap_auroc(y_te, best['score'])
        results[f'sae_L{L}'] = {'feature': best['feat'], 'auroc_point': best['auroc_pt'],
                                  'ci_lo': lo, 'ci_hi': hi, 'sep': best['sep']}
        print(f'  SAE L{L} f{best["feat"]:>5d}: {best["auroc_pt"]:.4f} [{lo:.3f}, {hi:.3f}]')
    # Linear probe + diff-means at L11/L31/L55 (paper compares at these layers)
    for L in SAE_LAYERS:
        X_tr = residuals_all[tr_global, L]; X_te = residuals_all[te_global, L]
        sc = StandardScaler().fit(X_tr)
        clf = LogisticRegression(C=1.0, max_iter=2000, random_state=SEED).fit(sc.transform(X_tr), y_tr)
        s_lp = clf.decision_function(sc.transform(X_te))
        m, lo, hi = bootstrap_auroc(y_te, s_lp)
        pt = roc_auc_score(y_te, s_lp)
        results[f'lp_L{L}'] = {'auroc_point': float(pt), 'ci_lo': lo, 'ci_hi': hi}
        print(f'  LR  L{L}: {pt:.4f} [{lo:.3f}, {hi:.3f}]')
        mu_k = X_tr[y_tr == 1].mean(0); mu_u = X_tr[y_tr == 0].mean(0)
        d = (mu_k - mu_u) / (np.linalg.norm(mu_k - mu_u) + 1e-12)
        s_dm = X_te @ d
        if roc_auc_score(y_te, s_dm) < 0.5: s_dm = -s_dm
        m, lo, hi = bootstrap_auroc(y_te, s_dm)
        pt = roc_auc_score(y_te, s_dm)
        results[f'dm_L{L}'] = {'auroc_point': float(pt), 'ci_lo': lo, 'ci_hi': hi}
        print(f'  DM  L{L}: {pt:.4f} [{lo:.3f}, {hi:.3f}]')
    return results

result_ferr = evaluate(y_ferr, 'ferrando_style')
result_ref  = evaluate(y_ref,  'refusal_only')

## 6. Compare + save

In [ ]:
comparison = {
    'description': 'Sensitivity analysis: AUROCs under refusal-only vs Ferrando-style (confabulation=unknown) labelling, on the same residual capture.',
    'rules': {
        'ferrando_style': 'correct>=2 -> known; correct==0 -> unknown (regardless of refusal); correct==1 -> middle',
        'refusal_only':   'correct>=2 -> known; correct==0 AND refused>=1 -> unknown; else -> middle',
    },
    'class_balance': {
        'ferrando_style': {
            'known':   sum(1 for x in ferrando if x['class']=='known'),
            'unknown': sum(1 for x in ferrando if x['class']=='unknown'),
            'middle':  sum(1 for x in ferrando if x['class']=='middle'),
        },
        'refusal_only': {
            'known':   sum(1 for x in refusal if x['class']=='known'),
            'unknown': sum(1 for x in refusal if x['class']=='unknown'),
            'middle':  sum(1 for x in refusal if x['class']=='middle'),
        },
    },
    'aurocs': {
        'ferrando_style': result_ferr,
        'refusal_only':   result_ref,
    },
}
out_path = '/content/sensitivity_refusal_vs_ferrando.json'
with open(out_path, 'w') as f:
    json.dump(comparison, f, indent=2)
HfApi().upload_file(
    path_or_fileobj=out_path,
    path_in_repo='paper_baselines/sensitivity_refusal_vs_ferrando.json',
    repo_id=HF_SAE_REPO,
    commit_message='Notebook 28b · sensitivity analysis — refusal-only vs Ferrando-style labelling',
)
print('\n=== HEADLINE COMPARISON ===')
for L in SAE_LAYERS:
    if result_ferr and result_ref:
        f_pt = result_ferr[f'sae_L{L}']['auroc_point']; f_lo = result_ferr[f'sae_L{L}']['ci_lo']; f_hi = result_ferr[f'sae_L{L}']['ci_hi']
        r_pt = result_ref[f'sae_L{L}']['auroc_point'];  r_lo = result_ref[f'sae_L{L}']['ci_lo'];  r_hi = result_ref[f'sae_L{L}']['ci_hi']
        print(f'  SAE L{L}: ferrando {f_pt:.3f} [{f_lo:.3f},{f_hi:.3f}]  vs  refusal-only {r_pt:.3f} [{r_lo:.3f},{r_hi:.3f}]  Δ={r_pt-f_pt:+.3f}')
        f_pt = result_ferr[f'lp_L{L}']['auroc_point'];  r_pt = result_ref[f'lp_L{L}']['auroc_point']
        print(f'  LR  L{L}: ferrando {f_pt:.3f}                  vs  refusal-only {r_pt:.3f}                  Δ={r_pt-f_pt:+.3f}')